# StudyAssistant Final Experiments and Ablation Studies

This notebook follows the final experimental design for the StudyAssistant RAG pipeline.

Final pipeline:

```text
PDF Extraction with PyMuPDF
-> Conservative Structure-Preserving Cleaning
-> Semantic Chunking
-> BM25 Lexical Index + Dense FAISS Index
-> Hybrid Min-Max Score Fusion
-> Cross-Encoder Reranking
-> Context Construction with Source Markers
-> Gemini Grounded Generation
-> Citation Validation
```

The notebook separates **main experiments** from **ablation studies**. Main experiments evaluate the final selected system. Ablations change one technical component at a time.


## 0. Imports


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from study_assistant.config import settings
from study_assistant.evaluation import load_benchmark_csv
from run_experiments import (
    DEFAULT_ALPHAS,
    DEFAULT_CHUNK_CONFIGS,
    DEFAULT_CHUNKING_STRATEGIES,
    DEFAULT_RERANK_INITIAL_KS,
    build_index,
    ensure_output_dir,
    run_main_retrieval,
    run_qa_generation,
    run_citation_validation_eval,
    run_learning_output_validation,
    run_error_analysis,
    run_ablation_chunking_strategy,
    run_ablation_chunk_size,
    run_ablation_retrieval_component,
    run_ablation_alpha,
    run_ablation_reranker,
)


## 1. Experimental Setup

Corpus A is the main lecture-slide PDF collection in `data/`. If a long-document corpus is available, change `args.data_dir` to that folder and rerun the same notebook.

The benchmark format is:

```csv
question,answer,relevant_pages,relevant_chunk_ids,question_type
```

If `relevant_chunk_ids` is empty, page-level relevance is used. A retrieved chunk is relevant when its `filename:page` matches a gold page label.


In [ ]:
args = SimpleNamespace(
    data_dir=ROOT / "data",
    benchmark=ROOT / "benchmarks" / "real_benchmark.csv",
    output_dir=ensure_output_dir(ROOT / "outputs" / "final_experiments"),

    # Final pipeline defaults.
    chunking_strategy="semantic",
    chunk_size=700,
    overlap=100,
    embedding_model=settings.embedding_model,
    bm25_tokenizer=settings.bm25_tokenizer,
    candidate_k=30,
    rerank_initial_k=30,
    top_k=5,
    alpha=0.7,

    # Ablation values.
    alphas=DEFAULT_ALPHAS,
    chunk_configs=DEFAULT_CHUNK_CONFIGS,
    chunking_strategies=DEFAULT_CHUNKING_STRATEGIES,
    rerank_initial_ks=DEFAULT_RERANK_INITIAL_KS,

    # Runtime controls.
    limit=0,
    run_generation=bool(os.getenv("GOOGLE_API_KEY")),
    run_reranker_ablation=False,
    google_api_key=os.getenv("GOOGLE_API_KEY", ""),
    quiz_count=settings.quiz_default_count,
    flashcard_count=settings.flashcard_default_count,
    error_limit=10,
    command="notebook",
)

vars(args)


## 2. Load Corpus, Benchmark, and Build Default Index

The default index uses the final document-processing configuration: PyMuPDF extraction, conservative cleaning, semantic chunking, BM25, dense embeddings, and FAISS `IndexFlatIP`.


In [ ]:
pdfs = sorted(args.data_dir.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No PDF files found in {args.data_dir}")

benchmark = load_benchmark_csv(args.benchmark)
if args.limit:
    benchmark = benchmark[: args.limit]

print(f"PDF files: {len(pdfs)}")
print(f"Benchmark questions: {len(benchmark)}")
display(pd.Series([row.get("question_type", "unknown") for row in benchmark]).value_counts().rename("count").to_frame())

index, index_time_sec = build_index(
    args.data_dir,
    args.chunk_size,
    args.overlap,
    args.embedding_model,
    args.bm25_tokenizer,
    args.chunking_strategy,
)
print(f"Built default index with {len(index.chunks)} chunks in {index_time_sec:.2f}s")


# 4. Main Experiments

Main experiments evaluate the final system after selecting the intended configuration. Component comparisons are kept in the ablation section.


## 4.1 Main Experiment 1: Final Retrieval Performance

Objective: evaluate the retrieval quality of the final retrieval pipeline.

Final retrieval pipeline:

```text
Semantic Chunking -> BM25 + Dense Retrieval -> Hybrid Min-Max Fusion -> Cross-Encoder Reranking -> Top-5 Chunks
```

Metrics: `Recall@5`, `MRR@5`, `Precision@5`, `nDCG@5`, and latency. The second table breaks results down by `question_type`.


In [ ]:
main_retrieval_outputs = run_main_retrieval(args, index, benchmark)

final_retrieval_df = main_retrieval_outputs["main_final_retrieval"]
final_retrieval_by_type_df = main_retrieval_outputs["main_final_retrieval_by_type"]

display(final_retrieval_df)
display(final_retrieval_by_type_df)


## 4.2 Main Experiment 2: End-to-End QA Generation

Objective: compare direct Gemini against the final RAG pipeline.

Compared systems:

```text
S1: Direct Gemini without retrieved context
S2: Final RAG Pipeline
```

Metrics: `Exact Match`, `Token F1`, `Answerability Accuracy`, `Citation Coverage`, `Citation Validity`, `Groundedness Score`, and latency.

This cell is disabled by default because it calls the Gemini API. Set `args.run_generation = True` and provide `GOOGLE_API_KEY` to run it.


In [ ]:
args.google_api_key = os.getenv("GOOGLE_API_KEY", "")
args.run_generation = bool(args.google_api_key)
args.limit = 1 if args.run_generation else 0

if not args.run_generation:
    print("Skipped generation setup. Set GOOGLE_API_KEY to run generation experiments.")


In [ ]:
if args.run_generation:
    qa_generation_df = run_qa_generation(args, index, benchmark)
    display(qa_generation_df)
else:
    print("Skipped QA generation. Set args.run_generation = True and configure GOOGLE_API_KEY to run this experiment.")


## 4.3 Main Experiment 3: Citation Validation Evaluation

Objective: test whether the citation validator detects missing, invalid, malformed, and out-of-context citation markers.

Test cases include valid citation `[S1]`, missing citation, invalid citation `[S9]`, malformed citation, and citation not included in the retrieved context.


In [ ]:
citation_validation_df = run_citation_validation_eval(args)
display(citation_validation_df)


## 4.4 Main Experiment 4: Learning Output Validation

Objective: evaluate whether the system can generate structured study materials from retrieved context.

Tasks:

```text
Summary generation
Quiz generation
Flashcard generation
```

Metrics: `JSON Success`, `Structure Validity`, `Duplicate Rate`, `Citation Validity`, and manual groundedness.

This cell is disabled by default because it calls the Gemini API. Set `args.run_generation = True` and provide `GOOGLE_API_KEY` to run it.


In [ ]:
if args.run_generation:
    learning_output_df = run_learning_output_validation(args, index, benchmark)
    display(learning_output_df)
else:
    print("Skipped learning output validation. Set args.run_generation = True and configure GOOGLE_API_KEY to run this experiment.")


## 4.5 Main Experiment 5: Error Analysis

Objective: inspect remaining failures of the final pipeline.

The table stores representative retrieval failures with gold sources, retrieved sources, an error type, cause, and possible fix. Generation and citation failures can be added manually after running API-based experiments.


In [ ]:
error_analysis_df = run_error_analysis(args, index, benchmark)
display(error_analysis_df)


# 5. Ablation Studies

Ablations evaluate the contribution of individual components. Unless stated otherwise, the fixed configuration is:

```text
PDF extraction: PyMuPDF
Cleaning: conservative structure-preserving cleaning
Chunking: semantic chunking
Chunk size: 700 words
Overlap: 100 words
Embedding model: paraphrase-multilingual-MiniLM-L12-v2
FAISS index: IndexFlatIP with normalized vectors
Fusion: Hybrid Min-Max
Final top-k: 5
```


## 5.1 Ablation 1: Chunking Strategy

Objective: compare naive word-window chunking, paragraph-aware chunking, and semantic chunking.

Fixed setting: conservative cleaning, chunk size 700, overlap 100, Hybrid Min-Max retrieval, reranker disabled, top-k 5.


In [ ]:
chunking_strategy_df = run_ablation_chunking_strategy(args, benchmark)
display(chunking_strategy_df)


## 5.2 Ablation 2: Chunk Size and Overlap

Objective: find a suitable chunk size and overlap for lecture PDF retrieval.

Compared variants:

```text
300 / 50
500 / 80
700 / 100
1000 / 150
1500 / 200
```


In [ ]:
chunk_size_df = run_ablation_chunk_size(args, benchmark)
display(chunk_size_df)


## 5.3 Ablation 3: Retrieval Component

Objective: evaluate the contribution of lexical retrieval, dense semantic retrieval, and hybrid fusion.

Compared variants:

```text
BM25 only
Dense only
Hybrid Min-Max Fusion
```

Reranking is disabled in this ablation.


In [ ]:
retrieval_component_df = run_ablation_retrieval_component(args, index, benchmark)
display(retrieval_component_df)


## 5.4 Ablation 4: Hybrid Alpha

Objective: tune the balance between BM25 score and dense retrieval score.

Hybrid score:

```text
score = alpha * dense_score + (1 - alpha) * bm25_score
```

`alpha = 0.0` is pure BM25, and `alpha = 1.0` is pure dense retrieval.


In [ ]:
alpha_df = run_ablation_alpha(args, index, benchmark)
display(alpha_df)

metric_cols = [col for col in [f"recall@{args.top_k}", f"mrr@{args.top_k}", f"ndcg@{args.top_k}"] if col in alpha_df.columns]
if metric_cols:
    plot_df = alpha_df.copy()
    plot_df["alpha"] = plot_df["method"].str.replace("alpha_", "", regex=False).astype(float)
    plot_df.plot(x="alpha", y=metric_cols, marker="o", figsize=(8, 4))
    plt.title("Hybrid Alpha Ablation")
    plt.xlabel("alpha")
    plt.ylabel("score")
    plt.grid(True, alpha=0.3)
    plt.show()


## 5.5 Ablation 5: Cross-Encoder Reranking

Objective: evaluate whether cross-encoder reranking improves final ranking quality.

Compared variants:

```text
Hybrid without reranking
Hybrid + rerank top 10 candidates -> top 5
Hybrid + rerank top 20 candidates -> top 5
Hybrid + rerank top 30 candidates -> top 5
Hybrid + rerank top 50 candidates -> top 5
```

This cell is disabled by default because reranking is slower. Set `args.run_reranker_ablation = True` to run it.


In [ ]:
if args.run_reranker_ablation:
    reranker_df = run_ablation_reranker(args, index, benchmark)
    display(reranker_df)
else:
    print("Skipped reranker ablation. Set args.run_reranker_ablation = True to run this ablation.")


# 6. Final Configuration Selection

Select the final configuration using these criteria:

```text
Primary criterion:
Highest or near-highest Recall@5, MRR@5, and nDCG@5.

Secondary criterion:
High citation validity and groundedness in end-to-end generation.

Practical criterion:
Latency is acceptable for the Streamlit demo.
```


In [ ]:
if "alpha_df" in globals() and not alpha_df.empty:
    ndcg_col = f"ndcg@{args.top_k}"
    if ndcg_col in alpha_df.columns:
        print("Best alpha by nDCG:")
        display(alpha_df.sort_values(ndcg_col, ascending=False).head(1))

if "chunking_strategy_df" in globals() and not chunking_strategy_df.empty:
    print("Chunking strategy results:")
    display(chunking_strategy_df)

if "chunk_size_df" in globals() and not chunk_size_df.empty:
    print("Chunk size results:")
    display(chunk_size_df)


# 7. Saved Output Files

All experiment tables are saved as CSV files under `outputs/final_experiments/`.


In [ ]:
for csv_path in sorted(args.output_dir.glob("*.csv")):
    print(csv_path.relative_to(ROOT))
